# Panel G no-ULC / no-Kinnex sensitivity run (HPC)

Goal: regenerate the Panel G transcript concordance counts after excluding CAT transcript models whose gene/transcript biotype is `unknown_likely_coding`.

This notebook is intended to run on the HPC filesystem where the original GFFs and per-assembly QC outputs are available. It does not modify the original pipeline outputs; it writes to a separate sensitivity directory.

Expected copy-back folder after completion: `results/intermediate_spreadsheets/kinnex_sensitivity/`.


In [ ]:
from pathlib import Path
import os, gzip, shutil, subprocess, json, textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = Path(os.getenv('HPRC_QC_REPO_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc'))
OUTPUT_DIR = Path(os.getenv('HPRC_QC_OUTPUT_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))
QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
INTERMEDIATE_DIR = OUTPUT_DIR / 'intermediate_spreadsheets'
ALL_INTRON_DIR = INTERMEDIATE_DIR / 'intron_chain'
WORK_DIR = INTERMEDIATE_DIR / 'kinnex_sensitivity'

ENSEMBL_CACHE_DIR = Path(os.getenv('HPRC_QC_ENSEMBL_CACHE_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/cache/ensembl'))
CAT_CACHE_DIR = Path(os.getenv('HPRC_QC_CAT_CACHE_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/cache/cat'))
ASSEMBLY_MAP_TSV = Path(os.getenv('HPRC_ASSEMBLY_MAP_TSV', str(REPO_DIR / 'data/exclusive_gene_analysis/assembly_overview.tsv')))

BIN_DIR = REPO_DIR / 'nextflow/pipelines/ensembl_cat_comparison/bin'
CALC_TC = BIN_DIR / 'calculate_transcript_concordance.py'
COUNT_TX = BIN_DIR / 'count_gff_transcripts.py'
AGG_INTRON = BIN_DIR / 'aggregate_intron_chain_by_biotype.py'

for p in [WORK_DIR, WORK_DIR/'filtered_cat_gff_no_ulc', WORK_DIR/'transcript_concordance_no_ulc', WORK_DIR/'cat_gene_counts_no_ulc', WORK_DIR/'intron_chain_no_ulc', WORK_DIR/'figures']:
    p.mkdir(parents=True, exist_ok=True)

print('REPO_DIR:', REPO_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('ASSEMBLY_MAP_TSV:', ASSEMBLY_MAP_TSV, ASSEMBLY_MAP_TSV.exists())
print('Scripts exist:', CALC_TC.exists(), COUNT_TX.exists(), AGG_INTRON.exists())


In [ ]:
# Helpers: GFF filtering and input resolution.
TRANSCRIPT_TYPES = {
    'transcript', 'mRNA', 'lnc_RNA', 'ncRNA', 'miRNA', 'snoRNA', 'snRNA', 'tRNA', 'rRNA',
    'pseudogenic_transcript', 'antisense_RNA', 'guide_RNA', 'scRNA', 'vault_RNA', 'Y_RNA'
}

ULC_VALUE = 'unknown_likely_coding'

def open_text(path, mode='rt'):
    path = str(path)
    return gzip.open(path, mode) if path.endswith('.gz') else open(path, mode)

def parse_attrs(attr):
    d = {}
    for item in attr.strip().split(';'):
        if '=' in item:
            k, v = item.split('=', 1)
            d[k] = v
    return d

def attr_biotype(attrs):
    return (attrs.get('biotype') or attrs.get('gene_biotype') or attrs.get('transcript_biotype') or '').strip()

def split_parent(parent):
    return [p for p in str(parent or '').split(',') if p]

def filter_cat_gff_no_ulc(in_gff, out_gff):
    """Remove unknown_likely_coding CAT genes/transcripts and their child features."""
    in_gff, out_gff = Path(in_gff), Path(out_gff)
    if out_gff.exists() and out_gff.stat().st_size > 0:
        return out_gff

    skip_genes = set()
    skip_txs = set()

    # Pass 1: identify ULC genes/transcripts.
    with open_text(in_gff, 'rt') as fh:
        for line in fh:
            if not line or line.startswith('#'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9:
                continue
            ftype = parts[2]
            attrs = parse_attrs(parts[8])
            bio = attr_biotype(attrs)
            fid = attrs.get('ID') or attrs.get('gene_id') or attrs.get('transcript_id')
            parent = attrs.get('Parent') or attrs.get('gene_id')
            if bio == ULC_VALUE:
                if ftype == 'gene' or ftype.endswith('gene'):
                    if fid:
                        skip_genes.add(fid)
                elif ftype in TRANSCRIPT_TYPES:
                    if fid:
                        skip_txs.add(fid)
                    for p in split_parent(parent):
                        if p:
                            skip_genes.add(p)

    # Pass 1b: any transcript under a skipped gene should also be skipped.
    with open_text(in_gff, 'rt') as fh:
        for line in fh:
            if not line or line.startswith('#'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9:
                continue
            ftype = parts[2]
            if ftype not in TRANSCRIPT_TYPES:
                continue
            attrs = parse_attrs(parts[8])
            fid = attrs.get('ID') or attrs.get('transcript_id')
            parents = split_parent(attrs.get('Parent') or attrs.get('gene_id'))
            if any(p in skip_genes for p in parents) and fid:
                skip_txs.add(fid)

    # Pass 2: write filtered GFF.
    opener = gzip.open if str(out_gff).endswith('.gz') else open
    with open_text(in_gff, 'rt') as inp, opener(out_gff, 'wt') as out:
        for line in inp:
            if not line or line.startswith('#'):
                out.write(line)
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9:
                out.write(line)
                continue
            ftype = parts[2]
            attrs = parse_attrs(parts[8])
            fid = attrs.get('ID') or attrs.get('gene_id') or attrs.get('transcript_id')
            parents = split_parent(attrs.get('Parent') or attrs.get('gene_id') or attrs.get('transcript_id'))
            if fid in skip_genes or fid in skip_txs or any(p in skip_genes or p in skip_txs for p in parents):
                continue
            out.write(line)

    return out_gff

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists() and p.stat().st_size > 0:
            return p
    return None

def resolve_inputs(accession, sample):
    accession = str(accession); sample = str(sample)
    ensembl_candidates = [
        ENSEMBL_CACHE_DIR / f'{accession}.gff3.gz',
        ENSEMBL_CACHE_DIR / f'{accession}.ensembl.renamed.gff3.gz',
        RESULTS_DIR / accession / f'{accession}.ensembl.renamed.gff3.gz',
    ] + sorted((RESULTS_DIR / accession).glob('*ensembl*.gff3.gz'))
    cat_candidates = [
        CAT_CACHE_DIR / f'{sample}_cat.gff3.gz',
        CAT_CACHE_DIR / f'{sample}_CAT_v2.0.gff3.gz',
        RESULTS_DIR / accession / f'{sample}_cat.gff3.gz',
    ] + sorted((RESULTS_DIR / accession).glob('*cat*.gff3.gz'))
    rbh_candidates = [RESULTS_DIR / accession / f'{accession}.gene_pairs_rbh.tsv'] + sorted((RESULTS_DIR / accession).glob('*gene_pairs_rbh.tsv'))
    return first_existing(ensembl_candidates), first_existing(cat_candidates), first_existing(rbh_candidates)


In [ ]:
# Build assembly run table.
assembly_map = pd.read_csv(ASSEMBLY_MAP_TSV, sep='\t', usecols=['assembly_accession','sample_name']).drop_duplicates()
records = []
for row in assembly_map.itertuples(index=False):
    ens_gff, cat_gff, rbh = resolve_inputs(row.assembly_accession, row.sample_name)
    records.append({
        'assembly_accession': row.assembly_accession,
        'sample_name': row.sample_name,
        'ensembl_gff': str(ens_gff) if ens_gff else '',
        'cat_gff': str(cat_gff) if cat_gff else '',
        'rbh_pairs': str(rbh) if rbh else '',
        'ready': bool(ens_gff and cat_gff and rbh),
    })
run_table = pd.DataFrame(records)
run_table.to_csv(WORK_DIR / 'panel_g_no_ulc_input_resolution.tsv', sep='\t', index=False)
print(run_table['ready'].value_counts(dropna=False))
display(run_table.head())
if not run_table['ready'].all():
    display(run_table[~run_table['ready']].head(20))


In [ ]:
# Run filtered transcript concordance and CAT gene transcript counts.
# This can take a while. It is resumable: existing non-empty output files are skipped.
ready = run_table[run_table['ready']].copy()
failed = []
for i, r in enumerate(ready.itertuples(index=False), 1):
    acc = r.assembly_accession
    sample = r.sample_name
    if i == 1 or i % 25 == 0:
        print(f'{i}/{len(ready)} {acc} {sample}', flush=True)

    filtered_cat = WORK_DIR / 'filtered_cat_gff_no_ulc' / f'{sample}_cat.no_unknown_likely_coding.gff3.gz'
    tc_out = WORK_DIR / 'transcript_concordance_no_ulc' / f'{acc}_transcript_concordance.tsv'
    cat_count_out = WORK_DIR / 'cat_gene_counts_no_ulc' / f'{acc}_cat_gene_transcript_counts.tsv'

    try:
        filter_cat_gff_no_ulc(r.cat_gff, filtered_cat)
        if not (tc_out.exists() and tc_out.stat().st_size > 0):
            subprocess.run([
                'python', str(CALC_TC),
                '--ensembl-gff', r.ensembl_gff,
                '--cat-gff', str(filtered_cat),
                '--pairs', r.rbh_pairs,
                '--output', str(tc_out),
                '--assembly-accession', acc,
                '--sample-name', sample,
            ], check=True)
        if not (cat_count_out.exists() and cat_count_out.stat().st_size > 0):
            subprocess.run([
                'python', str(COUNT_TX),
                '--gff', str(filtered_cat),
                '--output', str(cat_count_out),
                '--assembly-accession', acc,
            ], check=True)
    except Exception as e:
        failed.append({'assembly_accession': acc, 'sample_name': sample, 'error': repr(e)})
        print('FAILED', acc, e, flush=True)

pd.DataFrame(failed).to_csv(WORK_DIR / 'panel_g_no_ulc_failures.tsv', sep='\t', index=False)
print('failures:', len(failed))


In [ ]:
# Aggregate into intron-chain full-denominator tables.
# Existing Ensembl gene transcript count files are read from QC_DIR recursively.
subprocess.run([
    'python', str(AGG_INTRON),
    '--transcript-concordance-dir', str(WORK_DIR / 'transcript_concordance_no_ulc'),
    '--output-dir', str(WORK_DIR / 'intron_chain_no_ulc'),
    '--all-ensembl-genes-dir', str(QC_DIR),
    '--all-cat-genes-dir', str(WORK_DIR / 'cat_gene_counts_no_ulc'),
], check=True)
print('Wrote:', sorted((WORK_DIR / 'intron_chain_no_ulc').glob('*.tsv')))


In [ ]:
# Render Panel G all-model vs no-ULC side by side.
BIOTYPE_ORDER = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA': 'lncRNA',
    'pseudogene': 'Pseudogene',
    'other_ncRNA': 'Other ncRNA',
    'other': 'Other coding types',
}
FULL_DENOM_GROUP_MAP = {
    'Exact_Match': 'Exact match',
    'Intron_Match': 'Same intron chain',
    'Intron_Subset': 'Partial overlap',
    'Intron_Superset': 'Partial overlap',
    'Partial_5': 'Partial overlap',
    'Partial_3': 'Partial overlap',
    'Other_Partial': 'Partial overlap',
    'No_Match': 'No match',
    'Gene_Not_Shared': 'Gene not shared',
}
GROUP_5_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match', 'Gene not shared']
GROUP_5_COLORS = {
    'Exact match': '#2ecc71',
    'Same intron chain': '#7fd37f',
    'Partial overlap': '#f1c40f',
    'No match': '#e74c3c',
    'Gene not shared': '#95a5a6',
}

def med5_counts(df, direction):
    tmp = df[df['direction'] == direction].copy()
    tmp['group'] = tmp['classification'].map(FULL_DENOM_GROUP_MAP)
    grouped = tmp.groupby(['assembly_accession','biotype','group'])['n_transcripts'].sum().reset_index()
    med = grouped.groupby(['biotype','group'])['n_transcripts'].median().reset_index()
    return med.pivot(index='biotype', columns='group', values='n_transcripts').reindex(index=BIOTYPE_ORDER, columns=GROUP_5_ORDER, fill_value=0)

def save_panel_g(fd_df, label, out_prefix):
    med_ens = med5_counts(fd_df, 'Ensembl_to_CAT')
    med_cat = med5_counts(fd_df, 'CAT_to_Ensembl')
    med = pd.concat([
        med_ens.reset_index().melt(id_vars='biotype', var_name='classification_group', value_name='median_transcript_count').assign(direction='Ensembl_to_CAT'),
        med_cat.reset_index().melt(id_vars='biotype', var_name='classification_group', value_name='median_transcript_count').assign(direction='CAT_to_Ensembl'),
    ], ignore_index=True)
    med.to_csv(WORK_DIR / f'{out_prefix}_median_counts.tsv', sep='\t', index=False)

    fig, axes = plt.subplots(1, len(BIOTYPE_ORDER), figsize=(15, 3.2), sharex=False)
    bar_height = 0.34
    y_ens = 0.5 - bar_height / 2
    y_cat = 0.5 + bar_height / 2
    for ax, biotype in zip(axes, BIOTYPE_ORDER):
        left_ens = 0
        left_cat = 0
        for grp in GROUP_5_ORDER:
            v_ens = med_ens.loc[biotype, grp]
            v_cat = med_cat.loc[biotype, grp]
            ax.barh(y_ens, v_ens, left=left_ens, height=bar_height, color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3)
            ax.barh(y_cat, v_cat, left=left_cat, height=bar_height, color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3, alpha=0.65)
            left_ens += v_ens
            left_cat += v_cat
        ax.set_title(BIOTYPE_LABELS[biotype], fontsize=9)
        ax.set_yticks([y_ens, y_cat]); ax.set_yticklabels(['Ens→CAT', 'CAT→Ens'], fontsize=7)
        ax.set_ylim(1.0, 0.0)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle(label, fontsize=11, fontweight='bold', x=0.01, ha='left')
    fig.supxlabel('Median number of transcripts across assemblies', y=0.02)
    handles = [mpatches.Patch(color=GROUP_5_COLORS[g], label=g) for g in GROUP_5_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=5, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.08, 1, 0.9])
    fig.savefig(WORK_DIR / 'figures' / f'{out_prefix}.png', dpi=300, bbox_inches='tight')
    fig.savefig(WORK_DIR / 'figures' / f'{out_prefix}.pdf', bbox_inches='tight')
    plt.show()

all_fd = pd.read_csv(ALL_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv', sep='\t')
no_ulc_fd = pd.read_csv(WORK_DIR / 'intron_chain_no_ulc' / 'intron_chain_full_denom_per_assembly.tsv', sep='\t')
all_fd.to_csv(WORK_DIR / 'panel_g_all_models_full_denom_source.tsv', sep='\t', index=False)
no_ulc_fd.to_csv(WORK_DIR / 'panel_g_no_unknown_likely_coding_full_denom_source.tsv', sep='\t', index=False)

save_panel_g(all_fd, 'Panel G: all CAT models', 'panel_g_all_models')
save_panel_g(no_ulc_fd, 'Panel G: CAT excluding unknown_likely_coding models', 'panel_g_no_unknown_likely_coding')
print('Copy back:', WORK_DIR)
